In [1]:
import sys
from pathlib import Path

print(sys.executable)
print(Path.cwd())

c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\.venv\Scripts\python.exe
c:\temp\python_learning\ml_projects\ml_projects_batch_01\02_telco_customer_churn\notebooks


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(7043, 21)

In [4]:
target_col = "Churn"
id_cols = ["customerID"]

y = df[target_col].map({
    "No": 0,
    "Yes": 1,
})

X = df.drop(columns=id_cols + [target_col])
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

X.shape, y.shape

((7043, 19), (7043,))

In [5]:
numeric_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

all_features = numeric_features + categorical_features

len(numeric_features), len(categorical_features), len(all_features), X.shape[1]

(4, 15, 19, 19)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5634, 19), (1409, 19), (5634,), (1409,))

In [7]:
split_distribution = pd.DataFrame({
    "full": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

split_distribution

,full,train,test
Churn,,,
0,0.73463,0.734647,0.734564
1,0.26537,0.265353,0.265436


In [8]:
numeric_preprocessor_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

numeric_preprocessor_unscaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor_scaled, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor_unscaled, numeric_features),
        ("cat", categorical_preprocessor, categorical_features),
    ]
)

In [9]:
model_specs = {
    "Logistic Regression balanced": {
        "preprocessor": preprocessor_scaled,
        "model": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    },
    "Logistic Regression": {
        "preprocessor": preprocessor_scaled,
        "model": LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    },
    "Gradient Boosting": {
        "preprocessor": preprocessor_unscaled,
        "model": GradientBoostingClassifier(
            random_state=42,
        ),
    },
}

model_specs

{'Logistic Regression balanced': {'preprocessor': ColumnTransformer(transformers=[('num',
                                   Pipeline(steps=[('imputer',
                                                    SimpleImputer(strategy='median')),
                                                   ('scaler', StandardScaler())]),
                                   ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                    'TotalCharges']),
                                  ('cat',
                                   Pipeline(steps=[('imputer',
                                                    SimpleImputer(strategy='most_frequent')),
                                                   ('onehot',
                                                    OneHotEncoder(handle_unknown='ignore'))]),
                                   ['gender', 'Partner', 'Dependents',
                                    'PhoneService', 'MultipleLines',
                                    'InternetS

In [10]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

Ячейка 10 — OOF predicted probabilities

In [11]:
oof_predictions = {}

for model_name, spec in model_specs.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", spec["preprocessor"]),
            ("model", spec["model"]),
        ]
    )
    
    proba = cross_val_predict(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    
    oof_predictions[model_name] = proba

list(oof_predictions.keys())

['Logistic Regression balanced', 'Logistic Regression', 'Gradient Boosting']

In [12]:
{k: v.shape for k, v in oof_predictions.items()}

{'Logistic Regression balanced': (5634,),
 'Logistic Regression': (5634,),
 'Gradient Boosting': (5634,)}

Добавь ячейки после oof_predictions.

Ячейка 11 — helper function for one threshold

In [13]:
def evaluate_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

Ячейка 12 — default threshold 0.5 comparison

In [14]:
default_threshold_results = []

for model_name, proba in oof_predictions.items():
    row = evaluate_threshold(
        y_true=y_train,
        y_proba=proba,
        threshold=0.5,
    )
    
    row["model"] = model_name
    row["roc_auc"] = roc_auc_score(y_train, proba)
    row["pr_auc"] = average_precision_score(y_train, proba)
    
    default_threshold_results.append(row)

default_threshold_df = pd.DataFrame(default_threshold_results)

default_threshold_df = default_threshold_df[
    [
        "model",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
].sort_values("f1", ascending=False)

default_threshold_df_rounded = default_threshold_df.copy()

metric_cols = [
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

default_threshold_df_rounded[metric_cols] = default_threshold_df_rounded[metric_cols].round(4)

default_threshold_df_rounded

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,Logistic Regression balanced,0.5,0.7481,0.5164,0.8013,0.6280,0.8455,0.6569,3017,1122,297,1198
1,Logistic Regression,0.5,0.8021,0.6527,0.5431,0.5929,0.8456,0.6583,3707,432,683,812
2,Gradient Boosting,0.5,0.8033,0.6600,0.5338,0.5902,0.8471,0.6643,3728,411,697,798


In [15]:
thresholds = np.arange(0.10, 0.91, 0.05)

thresholds

array([0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55, 0.6 ,
       0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 ])

Ячейка 14 — threshold table for all models

In [17]:
threshold_results = []

for model_name, proba in oof_predictions.items():
    roc_auc = roc_auc_score(y_train, proba)
    pr_auc = average_precision_score(y_train, proba)
    
    for threshold in thresholds:
        row = evaluate_threshold(
            y_true=y_train,
            y_proba=proba,
            threshold=threshold,
        )
        
        row["model"] = model_name
        row["roc_auc"] = roc_auc
        row["pr_auc"] = pr_auc
        
        threshold_results.append(row)

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df[
    [
        "model",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
]

threshold_df_rounded = threshold_df.copy()
threshold_df_rounded[metric_cols] = threshold_df_rounded[metric_cols].round(4)

threshold_df_rounded.head()

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,Logistic Regression balanced,0.10,0.4828,0.3370,0.9813,0.5017,0.8455,0.6569,1253,2886,28,1467
1,Logistic Regression balanced,0.15,0.5422,0.3640,0.9706,0.5295,0.8455,0.6569,1604,2535,44,1451
2,Logistic Regression balanced,0.20,0.5850,0.3860,0.9545,0.5497,0.8455,0.6569,1869,2270,68,1427
3,Logistic Regression balanced,0.25,0.6170,0.4046,0.9405,0.5658,0.8455,0.6569,2070,2069,89,1406
4,Logistic Regression balanced,0.30,0.6470,0.4243,0.9258,0.5819,0.8455,0.6569,2261,1878,111,1384


Ячейка 15 — best threshold by F1 for each model

In [18]:
best_threshold_by_f1 = (
    threshold_df
    .sort_values(["model", "f1"], ascending=[True, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

best_threshold_by_f1_rounded = best_threshold_by_f1.copy()
best_threshold_by_f1_rounded[metric_cols] = best_threshold_by_f1_rounded[metric_cols].round(4)

best_threshold_by_f1_rounded.sort_values("f1", ascending=False)

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
2,Logistic Regression balanced,0.60,0.7822,0.5712,0.7191,0.6367,0.8455,0.6569,3332,807,420,1075
1,Logistic Regression,0.35,0.7817,0.5705,0.7171,0.6354,0.8456,0.6583,3332,807,423,1072
0,Gradient Boosting,0.35,0.7852,0.5783,0.7043,0.6351,0.8471,0.6643,3371,768,442,1053


Ячейка 16 — reasonable threshold range for churn detection

Для churn detection нам может быть интересен не только max F1, но и область, где recall остаётся высоким.

In [19]:
high_recall_thresholds = (
    threshold_df
    .loc[threshold_df["recall"] >= 0.70]
    .sort_values(["model", "threshold"])
    .copy()
)

high_recall_thresholds_rounded = high_recall_thresholds.copy()
high_recall_thresholds_rounded[metric_cols] = high_recall_thresholds_rounded[metric_cols].round(4)

high_recall_thresholds_rounded

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
34,Gradient Boosting,0.10,0.6211,0.4068,0.9338,0.5667,0.8471,0.6643,2103,2036,99,1396
35,Gradient Boosting,0.15,0.6834,0.4512,0.8943,0.5998,0.8471,0.6643,2513,1626,158,1337
36,Gradient Boosting,0.20,0.7203,0.4846,0.8508,0.6175,0.8471,0.6643,2786,1353,223,1272
37,Gradient Boosting,0.25,0.7446,0.5120,0.8020,0.6250,0.8471,0.6643,2996,1143,296,1199
38,Gradient Boosting,0.30,0.7675,0.5445,0.7565,0.6333,0.8471,0.6643,3193,946,364,1131
39,Gradient Boosting,0.35,0.7852,0.5783,0.7043,0.6351,0.8471,0.6643,3371,768,442,1053
17,Logistic Regression,0.10,0.6100,0.4002,0.9418,0.5617,0.8456,0.6583,2029,2110,87,1408
18,Logistic Regression,0.15,0.6663,0.4382,0.9124,0.5920,0.8456,0.6583,2390,1749,131,1364
19,Logistic Regression,0.20,0.7109,0.4751,0.8548,0.6108,0.8456,0.6583,2727,1412,217,1278
20,Logistic Regression,0.25,0.7428,0.5096,0.8167,0.6276,0.8456,0.6583,2964,1175,274,1221


In [20]:
high_recall_summary = (
    high_recall_thresholds
    .groupby("model")
    .agg(
        min_threshold=("threshold", "min"),
        max_threshold=("threshold", "max"),
        best_f1_in_high_recall=("f1", "max"),
        max_precision_in_high_recall=("precision", "max"),
        min_recall_in_high_recall=("recall", "min"),
        max_recall_in_high_recall=("recall", "max"),
    )
    .sort_values("best_f1_in_high_recall", ascending=False)
)

high_recall_summary.round(4)

,min_threshold,max_threshold,best_f1_in_high_recall,max_precision_in_high_recall,min_recall_in_high_recall,max_recall_in_high_recall
model,,,,,,
Logistic Regression balanced,0.1,0.60,0.6367,0.5712,0.7191,0.9813
Logistic Regression,0.1,0.35,0.6354,0.5705,0.7171,0.9418
Gradient Boosting,0.1,0.35,0.6351,0.5783,0.7043,0.9338


Ячейка 17 — focus tables for each model

In [21]:
for model_name in model_specs.keys():
    display(
        threshold_df_rounded
        .loc[threshold_df_rounded["model"] == model_name]
        [["model", "threshold", "precision", "recall", "f1", "fp", "fn", "tp"]]
        .sort_values("threshold")
    )

,model,threshold,precision,recall,f1,fp,fn,tp
0,Logistic Regression balanced,0.10,0.3370,0.9813,0.5017,2886,28,1467
1,Logistic Regression balanced,0.15,0.3640,0.9706,0.5295,2535,44,1451
2,Logistic Regression balanced,0.20,0.3860,0.9545,0.5497,2270,68,1427
3,Logistic Regression balanced,0.25,0.4046,0.9405,0.5658,2069,89,1406
4,Logistic Regression balanced,0.30,0.4243,0.9258,0.5819,1878,111,1384
5,Logistic Regression balanced,0.35,0.4460,0.9023,0.5969,1676,146,1349
6,Logistic Regression balanced,0.40,0.4707,0.8649,0.6096,1454,202,1293
7,Logistic Regression balanced,0.45,0.4915,0.8334,0.6184,1289,249,1246
8,Logistic Regression balanced,0.50,0.5164,0.8013,0.6280,1122,297,1198
9,Logistic Regression balanced,0.55,0.5429,0.7659,0.6354,964,350,1145


,model,threshold,precision,recall,f1,fp,fn,tp
17,Logistic Regression,0.10,0.4002,0.9418,0.5617,2110,87,1408
18,Logistic Regression,0.15,0.4382,0.9124,0.5920,1749,131,1364
19,Logistic Regression,0.20,0.4751,0.8548,0.6108,1412,217,1278
20,Logistic Regression,0.25,0.5096,0.8167,0.6276,1175,274,1221
21,Logistic Regression,0.30,0.5408,0.7666,0.6342,973,349,1146
22,Logistic Regression,0.35,0.5705,0.7171,0.6354,807,423,1072
23,Logistic Regression,0.40,0.5940,0.6635,0.6269,678,503,992
24,Logistic Regression,0.45,0.6236,0.6007,0.6119,542,597,898
25,Logistic Regression,0.50,0.6527,0.5431,0.5929,432,683,812
26,Logistic Regression,0.55,0.6852,0.4863,0.5689,334,768,727


,model,threshold,precision,recall,f1,fp,fn,tp
34,Gradient Boosting,0.10,0.4068,0.9338,0.5667,2036,99,1396
35,Gradient Boosting,0.15,0.4512,0.8943,0.5998,1626,158,1337
36,Gradient Boosting,0.20,0.4846,0.8508,0.6175,1353,223,1272
37,Gradient Boosting,0.25,0.5120,0.8020,0.6250,1143,296,1199
38,Gradient Boosting,0.30,0.5445,0.7565,0.6333,946,364,1131
39,Gradient Boosting,0.35,0.5783,0.7043,0.6351,768,442,1053
40,Gradient Boosting,0.40,0.6102,0.6462,0.6277,617,529,966
41,Gradient Boosting,0.45,0.6383,0.5913,0.6139,501,611,884
42,Gradient Boosting,0.50,0.6600,0.5338,0.5902,411,697,798
43,Gradient Boosting,0.55,0.6844,0.4569,0.5479,315,812,683


## Stage 5 threshold analysis conclusions

### Evaluation setup

- Threshold analysis was performed only on `X_train` / `y_train`.
- `X_test` / `y_test` were not evaluated.
- Out-of-fold predicted probabilities were generated with 5-fold `StratifiedKFold`.
- Preprocessing was inside sklearn `Pipeline` / `ColumnTransformer`.
- Imputers, scalers and encoders were fitted only inside CV folds.
- No large hyperparameter tuning was performed.
- Thresholds were analyzed using OOF probabilities only.

### Models compared

- `LogisticRegression(class_weight="balanced")`
- `LogisticRegression`
- `GradientBoostingClassifier`

### Default threshold 0.5

At threshold 0.5, the best model by F1 is:

- `LogisticRegression(class_weight="balanced")`
- F1 ≈ 0.6280
- precision ≈ 0.5164
- recall ≈ 0.8013

This model catches the most churn customers at the default threshold, but it also creates many false positives.

### Ranking metrics

Best model by PR-AUC:

- `GradientBoostingClassifier`
- PR-AUC ≈ 0.6643

Best model by ROC-AUC:

- `GradientBoostingClassifier`
- ROC-AUC ≈ 0.8471

This suggests that Gradient Boosting ranks churn risk slightly better, even though it is not the best model by F1 at threshold 0.5.

### Best threshold by OOF F1

Best OOF F1 results:

- `LogisticRegression(class_weight="balanced")`
  - threshold ≈ 0.60
  - precision ≈ 0.5712
  - recall ≈ 0.7191
  - F1 ≈ 0.6367

- `LogisticRegression`
  - threshold ≈ 0.35
  - precision ≈ 0.5705
  - recall ≈ 0.7171
  - F1 ≈ 0.6354

- `GradientBoostingClassifier`
  - threshold ≈ 0.35
  - precision ≈ 0.5783
  - recall ≈ 0.7043
  - F1 ≈ 0.6351

The three candidates are very close by best OOF F1.

### Precision / recall trade-off

- Lower threshold increases recall but reduces precision.
- Higher threshold increases precision but reduces recall.
- For churn detection, a reasonable threshold should depend on retention budget and the relative cost of false positives vs. false negatives.

### Reasonable threshold ranges for churn detection

If the goal is to keep recall at or above about 0.70, reasonable threshold ranges are:

- `LogisticRegression(class_weight="balanced")`: around 0.55–0.60
- `LogisticRegression`: around 0.30–0.35
- `GradientBoostingClassifier`: around 0.30–0.35

Very low thresholds such as 0.10 are probably too aggressive for most business settings because they create too many false positives.

### Candidate refinement

Recommended candidates for later final evaluation:

1. `LogisticRegression(class_weight="balanced")`
   - best at default threshold;
   - best OOF F1 after threshold adjustment;
   - strong recall-oriented baseline.

2. `LogisticRegression`
   - nearly same best OOF F1 after lowering threshold;
   - simpler class-weight-free alternative.

3. `GradientBoostingClassifier`
   - best PR-AUC and ROC-AUC;
   - may be useful if ranking customers by churn risk is more important than default-threshold classification.

### Important limitation

The best threshold found on OOF predictions is still an estimate from training data only.
It is useful for candidate refinement, but it is not final evidence of performance on unseen data.
Final performance must be checked later on the untouched test set only once the final candidate and threshold policy are fixed.